Question 3<br>
Section (a, b)

In [ ]:
import cv2
import numpy as np


DISPLAY_HEIGHT = 700

# Load images
img1 = cv2.imread("c1.jpg")
img2 = cv2.imread("c2.jpg")

if img1 is None or img2 is None:
    print("Error loading images")
    exit()

h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]

# Resize both images to same height (for display only)
scale1 = DISPLAY_HEIGHT / h1
scale2 = DISPLAY_HEIGHT / h2

img1_disp = cv2.resize(img1, (int(w1 * scale1), DISPLAY_HEIGHT))
img2_disp = cv2.resize(img2, (int(w2 * scale2), DISPLAY_HEIGHT))

combined = np.hstack((img1_disp, img2_disp))
split_x = img1_disp.shape[1]

# Keep Clicks
pts1 = []
pts2 = []

WINDOW = "LEFT=c1 (green) | RIGHT=c2 (red) | Press q when done"

def click_event(event, x, y, flags, param):
    global pts1, pts2, combined

    if event == cv2.EVENT_LBUTTONDOWN:

        if x < split_x:
            # LEFT image
            x1 = int(x / scale1)
            y1 = int(y / scale1)

            pts1.append([x1, y1])
            cv2.circle(combined, (x, y), 5, (0, 255, 0), -1)
            print(f"c1: ({x1}, {y1})")

        else:
            # RIGHT image
            x2_disp = x - split_x

            x2 = int(x2_disp / scale2)
            y2 = int(y / scale2)

            pts2.append([x2, y2])
            cv2.circle(combined, (x, y), 5, (0, 0, 255), -1)
            print(f"c2: ({x2}, {y2})")

        cv2.imshow(WINDOW, combined)


# UI
cv2.imshow(WINDOW, combined)
cv2.setMouseCallback(WINDOW, click_event)

print("\nClick corresponding points (same order!)")
print("Press 'q' when done\n")

while True:
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

# Validate
pts1 = np.array(pts1, dtype=np.float32)
pts2 = np.array(pts2, dtype=np.float32)

print(f"\nPoints c1: {len(pts1)}")
print(f"Points c2: {len(pts2)}")

if len(pts1) < 6 or len(pts2) < 6:
    print("Need at least 6 points!")
    exit()

if len(pts1) != len(pts2):
    print("Point counts do not match!")
    exit()

####################
# (a) Homography
####################
H, mask = cv2.findHomography(pts2, pts1)

np.set_printoptions(precision=4, suppress=True)

print("\nHomography Matrix:\n", H)

if mask is not None:
    print(f"Inliers: {np.sum(mask)} / {len(mask)}")

# Warp
warped = cv2.warpPerspective(img2, H, (w1, h1))

# Resize for display
scale_warp = DISPLAY_HEIGHT / warped.shape[0]
warped_disp = cv2.resize(warped, (int(warped.shape[1]*scale_warp), DISPLAY_HEIGHT))

cv2.imshow("Warped Image", warped_disp)
cv2.waitKey(0)
cv2.destroyAllWindows()

#####################
# (b) Difference
#####################
diff = cv2.absdiff(img1, warped)

gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

# Resize for display
diff_disp = cv2.resize(diff, (int(diff.shape[1]*scale_warp), DISPLAY_HEIGHT))
thresh_disp = cv2.resize(thresh, (int(thresh.shape[1]*scale_warp), DISPLAY_HEIGHT))

cv2.imshow("Difference", diff_disp)
cv2.imshow("Threshold", thresh_disp)

cv2.waitKey(0)
cv2.destroyAllWindows()


Click corresponding points (same order!)
Press 'q' when done

c1: (966, 206)
c1: (1228, 292)
c1: (1672, 280)
c1: (2023, 584)
c1: (1294, 2565)
c1: (245, 2167)
c2: (1290, 75)
c2: (1508, 228)
c2: (1945, 328)
c2: (2209, 722)
c2: (983, 2435)
c2: (71, 1780)

Points c1: 6
Points c2: 6

Homography Matrix:
 [[   0.9591    0.2584 -284.7891]
 [  -0.2616    0.9597  470.0001]
 [  -0.       -0.        1.    ]]
Inliers: 2 / 6


Section (c)

In [24]:
import cv2
import numpy as np
import tkinter as tk

# Capture screen size
root = tk.Tk()
SCREEN_W = root.winfo_screenwidth()
SCREEN_H = root.winfo_screenheight()
root.destroy()

def show_fit(window_name, img):
    h, w = img.shape[:2]
    scale = min(SCREEN_W / w, SCREEN_H / h, 1.0) * 0.9
    resized = cv2.resize(img, (int(w * scale), int(h * scale)))

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.imshow(window_name, resized)

# Load images (grayscale)
img1 = cv2.imread("c1.jpg", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("c2.jpg", cv2.IMREAD_GRAYSCALE)

if img1 is None or img2 is None:
    print("Error loading images")
    exit()

# SIFT detector
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

print(f"Keypoints in c1: {len(kp1)}")
print(f"Keypoints in c2: {len(kp2)}")

# Match descriptors
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)

matches = bf.match(des1, des2)
print(f"Total matches: {len(matches)}")

# Sort matches
matches = sorted(matches, key=lambda x: x.distance)

# Limit matches for cleaner display
good_matches = matches[:50]

# Draw matches
matched_img = cv2.drawMatches(
    img1, kp1,
    img2, kp2,
    good_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

# Display Matches
show_fit("SIFT Matches (No Ratio Test)", matched_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

Keypoints in c1: 12312
Keypoints in c2: 11652
Total matches: 8527


Section (d)

In [26]:
import cv2
import numpy as np
import tkinter as tk

# Capture screen size
root = tk.Tk()
SCREEN_W = root.winfo_screenwidth()
SCREEN_H = root.winfo_screenheight()
root.destroy()

def show_fit(window_name, img):
    h, w = img.shape[:2]
    scale = min(SCREEN_W / w, SCREEN_H / h, 1.0) * 0.9
    resized = cv2.resize(img, (int(w * scale), int(h * scale)))
    cv2.imshow(window_name, resized)

# Load images
img1 = cv2.imread("c1.jpg")  # goldstandard
img2 = cv2.imread("c2.jpg")  # part removed

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# SIFT keypoints + descriptors
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print(f"Keypoints c1: {len(kp1)}")
print(f"Keypoints c2: {len(kp2)}")

# Match descriptors (KNN)
bf = cv2.BFMatcher(cv2.NORM_L2)

matches = bf.match(des1, des2)

# Sort matches
matches = sorted(matches, key=lambda x: x.distance)

# Limit matches for cleaner display
good_matches = matches[:50]

# Display matches
match_img = cv2.drawMatches(
    img1, kp1,
    img2, kp2,
    good_matches[:50],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

show_fit("SIFT Matches", match_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

#####################################
# (d) Compute homography from matches
#####################################

# Extract matched points
pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

# Compute homography (RANSAC!)
H, mask = cv2.findHomography(pts2, pts1, cv2.RANSAC, 5.0)

np.set_printoptions(precision=4, suppress=True)

print("\nHomography Matrix (SIFT):\n", H)
print(f"Inliers: {np.sum(mask)} / {len(mask)}")

# Warp image
h1, w1 = img1.shape[:2]
warped = cv2.warpPerspective(img2, H, (w1, h1))

show_fit("Warped (SIFT)", warped)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Difference
diff = cv2.absdiff(img1, warped)

gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

show_fit("Difference (SIFT)", diff)
show_fit("Threshold (SIFT)", thresh)

cv2.waitKey(0)
cv2.destroyAllWindows()

Keypoints c1: 12315
Keypoints c2: 11651

Homography Matrix (SIFT):
 [[   0.9643    0.2637 -290.3217]
 [  -0.2639    0.9642  478.0519]
 [  -0.       -0.        1.    ]]
Inliers: 50 / 50
